<h1>Bertopic Analysis</h1>

In this notebook, we will do bertopic analysis on newly collected data which was collected focusing on the english videos only.

<h2>Bertopic</h2>

In [1]:
import pandas as pd

df = pd.read_csv('clean_data.csv', keep_default_na=False)

In [2]:
df['text'] = df['title'] + ' ' + df['description']

In [7]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

model = SentenceTransformer("paraphrase-mpnet-base-v2") #more nuanced model (good for short texts) 

# Custom preprocessing with vectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",  
    ngram_range=(1, 2),    
    max_df=0.85,           
    min_df=2,              
    max_features=3000  
)

# Creating BERTopic model
topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=vectorizer_model,
    nr_topics="auto",
    top_n_words=10,       
    calculate_probabilities=True,
    min_topic_size=20 #reduces overly small clusters
)

In [9]:
topics, probs = topic_model.fit_transform(df['text'].fillna("").tolist())

In [11]:
tpp = topic_model.get_topic_info()

In [13]:
df['topic']=topics

#function to check if the video is related or unrelated

def is_video_related(text, keywords):
    text = str(text).lower()
    return any(kw in text for kw in keywords)

def check_relevance_ratio(df, topic_id, keywords):
    df_topic = df[df['topic']==topic_id]
    relevant_count = df_topic['text'].apply(lambda x: is_video_related(x, keywords)).sum()
    return relevant_count/len(df_topic)

keywords = ["longevity", "healthy aging", "biohacking", "rejuvenation", "anti-aging", "nootropics", "health"]

tpp['relevance_ratio'] = tpp['Topic'].apply(lambda t: check_relevance_ratio(df, t, keywords)) 

In [15]:
tpp

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio
0,-1,6194,-1_aging_anti_healthy_anti aging,"[aging, anti, healthy, anti aging, healthy agi...",[Dr. Dennis Gross Anti-Aging Scalp Serum John ...,0.706652
1,0,5891,0_aging_longevity_rejuvenation_anti,"[aging, longevity, rejuvenation, anti, anti ag...",[McLauren Anti-Aging Sunscreen Spray McLauren ...,0.790528
2,1,1987,1_biohacking_biohacker_biohack_dave,"[biohacking, biohacker, biohack, dave, dave as...",[#ShutYourMouth #HostageTape #PatrickMackeown ...,0.770005
3,2,1034,2_nootropics_nootropic_cortex_brain,"[nootropics, nootropic, cortex, brain, stack, ...",[Too many Nootropics in one stack for BEGINNER...,0.690522
4,3,67,3_klaus_mutant_workout_denmark,"[klaus, mutant, workout, denmark, 2012, super,...",[MUTANT athlete KLAUS RIIS will be at FIBO 201...,0.000000
5,4,61,4_dens_lower_song_nootropic,"[dens, lower, song, nootropic, music, record, ...",[Lower Dens - Propagation After just 76 second...,0.377049
6,5,52,5_arduino_nootropicdesign_nootropicdesign com_...,"[arduino, nootropicdesign, nootropicdesign com...",[TV Blaster Blast your TV with the Video Exper...,0.038462
7,6,37,6_video demonstrating_exercise video_demonstra...,"[video demonstrating, exercise video, demonstr...",[REJUVENATION | TRICEP EXTENSIONS Rejuvenation...,0.675676
8,7,35,7_diy_diy bio_bio_pellet,"[diy, diy bio, bio, pellet, water, outside, pr...","[DIY bio pellets Gambitz tank 08June., DIY Bio...",0.085714
9,8,34,8_big think_big_https bigth_bigth ink,"[big think, big, https bigth, bigth ink, bigth...",[What is your counsel? New videos DAILY: https...,0.117647


Let's check the topics with less than 40 percent relevance. Let's add also "is_relevant" column for both topics dataframe as well as for the data itslef. We will set it false if the topic and document is not related to our topic. 

The topics which has less than the 40 percent relevance are 3, 4, 5, 7, 8, 9. Let's aslo check 10 since it seems like it is about the welder tools which is not relevant to our topic. 

<h3> Manual check </h3>

In [37]:
#Let's set 'is_relevant' to True first
#After we will change it
tpp['is_relevant']=True
df['is_relevant']=True

i_topics = []  #irrelevant topics

In [39]:
#Let's check topic 3
df[df['topic']==3].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
69,YouTube,dljjppt2tJY,https://www.youtube.com/watch?v=dljjppt2tJY,Klaus Riis doing splits 2010,February 2010....Klaus doing splits while prac...,Klaus Myren Riis Viking Fitness Coach & Biohac...,WOW!! AWESOME!\n || haha ja skal jeg nok Mikke...,4298.0,7.0,3.0,biohacking,unknown,2010-02-02,False,Klaus Riis doing splits 2010 February 2010.......,3,True
77,YouTube,cRvG2NM0tFU,https://www.youtube.com/watch?v=cRvG2NM0tFU,Klaus Riis Posing seminar,Fisrt Posing Seminar held by Klaus Riis at Key...,Klaus Myren Riis Viking Fitness Coach & Biohac...,ham i army bukserne er da lidt af en frækkert ...,2441.0,3.0,1.0,biohacking,unknown,2010-02-08,False,Klaus Riis Posing seminar Fisrt Posing Seminar...,3,True
131,YouTube,YUGyUbbnjkc,https://www.youtube.com/watch?v=YUGyUbbnjkc,Klaus Riis Workout shoot 2010 sponsored by LOADED,Klaus Riis Workout shoot 2010 sponsored by LOA...,Klaus Myren Riis Viking Fitness Coach & Biohac...,"Jeg ved ikk meget om dig og din baggrund, men ...",2272.0,3.0,1.0,biohacking,unknown,2010-03-04,False,Klaus Riis Workout shoot 2010 sponsored by LOA...,3,True
426,YouTube,07b2RtHLClQ,https://www.youtube.com/watch?v=07b2RtHLClQ,TEASER - Training series with Klaus Riis,Teaser for Upcoming Training series featuring ...,Klaus Myren Riis Viking Fitness Coach & Biohac...,det ser sku godt ud!,2749.0,5.0,1.0,biohacking,en,2010-07-14,False,TEASER - Training series with Klaus Riis Tease...,3,True
435,YouTube,4X_v3mZKQ5Y,https://www.youtube.com/watch?v=4X_v3mZKQ5Y,Training series with Klaus Riis #1,Calves and Shoulders... Training series featur...,Klaus Myren Riis Viking Fitness Coach & Biohac...,de 3 andre dage? du er vel klar over at der er...,9502.0,20.0,4.0,biohacking,unknown,2010-07-16,False,Training series with Klaus Riis #1 Calves and ...,3,True


It is mostly about how the sportsman claled Klaus Riis is doing his workouts. Totally irelevant to our data. 

In [45]:
i_topics.append(3)

#let's check topic # 4
df[df['topic']==4].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
232,YouTube,oqMoseyUQio,https://www.youtube.com/watch?v=oqMoseyUQio,Rich Hardesty with Zuga,Rich has concert on the sands of Negril in fro...,Rich Hardesty,,192.0,3.0,0.0,biohacking,en,2010-04-25,False,Rich Hardesty with Zuga Rich has concert on th...,4,True
345,YouTube,cgnCIQPgPAs,https://www.youtube.com/watch?v=cgnCIQPgPAs,12-string Guitar: With My Swag All On My Shoul...,This is one of the songs of The Seekers that d...,threelegsoman,@ErunorLRugby Sounds like an interesting parody.,1971.0,16.0,1.0,biohacking,unknown,2010-06-15,False,12-string Guitar: With My Swag All On My Shoul...,4,True
371,YouTube,NbQbkzp9S4o,https://www.youtube.com/watch?v=NbQbkzp9S4o,Jiletta Riley The Way It Was,Vocal house.,SOULBISCUITS,🔥😈,3106.0,28.0,1.0,nootropics,en,2010-06-29,False,Jiletta Riley The Way It Was Vocal house.,4,True
401,YouTube,le9nIGk-EMM,https://www.youtube.com/watch?v=le9nIGk-EMM,Bassrk - Super Symmetry,Out now on Ammunition Recordings ! Buy it here...,NeurofunkGrid,steampunk machine gun FTW! ka clang! ka clang!...,7121.0,60.0,6.0,biohacking,en,2010-07-06,False,Bassrk - Super Symmetry Out now on Ammunition ...,4,True
428,YouTube,OcFr6C3rYg4,https://www.youtube.com/watch?v=OcFr6C3rYg4,Screw Face - Give It To Ya Raw,1995.,OGDonNinja,brutal || Screw face coming at y’all!!! || Suc...,4771.0,59.0,10.0,nootropics,unknown,2010-07-15,False,Screw Face - Give It To Ya Raw 1995.,4,True


This topic contains musics, which are totally unrelated to what we are doing

In [48]:
i_topics.append(4)

#let's check topic # 5
df[df['topic']==5].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
209,YouTube,1sdiTyklB8Y,https://www.youtube.com/watch?v=1sdiTyklB8Y,Ten Second War Trailer,Ten Second War is an indie video game that I'v...,AlmostMatt,Holy--------<br>the time limit idea is really ...,12011.0,24.0,5.0,biohacking,unknown,2010-04-10,False,Ten Second War Trailer Ten Second War is an in...,5,True
381,YouTube,N_6bZqhmxWQ,https://www.youtube.com/watch?v=N_6bZqhmxWQ,CPCO3 - MilkyMist One Interactive VJ Station,"La Milkymist One es una ""interactive VJ statio...",Campus Party,,1943.0,10.0,0.0,biohacking,en,2010-06-30,False,CPCO3 - MilkyMist One Interactive VJ Station L...,5,True
397,YouTube,2I0Z3PK54Ak,https://www.youtube.com/watch?v=2I0Z3PK54Ak,Example 12.1,This is older video content from the https://t...,tronixstuff,,7113.0,1.0,0.0,nootropics,unknown,2010-07-04,False,Example 12.1 This is older video content from ...,5,True
513,YouTube,dNE-AASFo5k,https://www.youtube.com/watch?v=dNE-AASFo5k,"Wii to DMX 512: A wii remote, a LED panel and ...",A wiimote does a wave effect and the nunchuk c...,matlightjams,,839.0,1.0,0.0,nootropics,unknown,2010-08-20,False,"Wii to DMX 512: A wii remote, a LED panel and ...",5,True
528,YouTube,xgqmulH1seo,https://www.youtube.com/watch?v=xgqmulH1seo,Solar Struggle - Intro English (Xbox 360 LIVE ...,,redspotgames,,128.0,1.0,0.0,biohacking,en,2010-08-27,False,Solar Struggle - Intro English (Xbox 360 LIVE ...,5,True


This topic also contains the videos which are totally unrelated to what we are doing

In [53]:
i_topics.append(5)

#let's check topic # 7
df[df['topic']==7].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
227,YouTube,dTz_b_mitlc,https://www.youtube.com/watch?v=dTz_b_mitlc,DIY Bio-Pellets Reactor,,Him .Cheng,,1026.0,0.0,0.0,biohacking,unknown,2010-04-23,False,DIY Bio-Pellets Reactor,7,True
297,YouTube,iU36n7C-3ik,https://www.youtube.com/watch?v=iU36n7C-3ik,Rickets Reef: DIY BioPellet Box,Quick vid showing a little box I made to hold ...,Rickets Reef,All you need to do is get a small screwdriver ...,12349.0,10.0,4.0,biohacking,en,2010-05-23,False,Rickets Reef: DIY BioPellet Box Quick vid show...,7,True
533,YouTube,36fWx3i5L4o,https://www.youtube.com/watch?v=36fWx3i5L4o,DIY Protein Skimmer by Sebastek - Panelowy Odp...,DIY Protein Skimmer by Sebastek - Panelowy Odp...,Sebastek74,seriously?... ANOTHER without narration!!!!......,7414.0,2.0,1.0,biohacking,unknown,2010-08-29,False,DIY Protein Skimmer by Sebastek - Panelowy Odp...,7,True
569,YouTube,RN4pMCi6_kg,https://www.youtube.com/watch?v=RN4pMCi6_kg,DiY Bio-Pellets reactor,"Cocacola lahev 2l 0,- Starší čerpadlo 1200 l/h...",S4Sulda,So how did that work out for you? Still in use...,5056.0,6.0,2.0,biohacking,unknown,2010-09-12,False,"DiY Bio-Pellets reactor Cocacola lahev 2l 0,- ...",7,True
782,YouTube,OVZwoNWcsrc,https://www.youtube.com/watch?v=OVZwoNWcsrc,New Design for a DIY Phosphate Reactor,This is the redesign if u dont understand how ...,GlobalWarmings,"if you want to know how to make it yourself , ...",18559.0,8.0,5.0,biohacking,unknown,2010-11-14,False,New Design for a DIY Phosphate Reactor This is...,7,True


This vidoes is also not connected to our research. It contains the vidoes which shows how to do some tools(e.g. reactor)

In [56]:
i_topics.append(7)

#let's check topic # 8
df[df['topic']==8].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
42,YouTube,H9AqAxq8-jY,https://www.youtube.com/watch?v=H9AqAxq8-jY,Study: Sitting &amp; Watching TV Can Kill You!,Follow us on Twitter: http://twitter.com/theyo...,The Young Turks,Thyme to go to go Home 🍪🍪🍪🍪🍪🍪 || even though y...,21914.0,511.0,154.0,biohacking,en,2010-01-23,False,Study: Sitting &amp; Watching TV Can Kill You!...,8,True
684,YouTube,zCmxqz5eMnc,https://www.youtube.com/watch?v=zCmxqz5eMnc,Eternal Vigilance and The Path Of Mastery,http://www.ericdobson.com Daily Podcast #005: ...,Eric Dobson,i&#39;ve been wanting to do something like thi...,79.0,2.0,1.0,nootropics,unknown,2010-10-24,False,Eternal Vigilance and The Path Of Mastery http...,8,True
1343,YouTube,DV3XjqW_xgU,https://www.youtube.com/watch?v=DV3XjqW_xgU,Michio Kaku: How to Reverse Aging | Big Think,New videos DAILY: https://bigth.ink Join Big T...,Big Think,"Want to get Smarter, Faster™?\r<br>Subscribe f...",2120423.0,25049.0,4264.0,well aging,en-US,2011-05-31,False,Michio Kaku: How to Reverse Aging | Big Think ...,8,True
1351,YouTube,5MeWBlKYOvw,https://www.youtube.com/watch?v=5MeWBlKYOvw,Neil deGrasse Tyson: Living and Longevity,New videos DAILY: https://bigth.ink Join Big T...,Big Think,Limited psychology about human potential. || I...,160718.0,2227.0,338.0,longevity,en-US,2011-06-03,False,Neil deGrasse Tyson: Living and Longevity New ...,8,True
1370,YouTube,X57lJXW0zL0,https://www.youtube.com/watch?v=X57lJXW0zL0,What Is Evil?,New videos DAILY: https://bigth.ink Join Big T...,Big Think,Evil is repeated sadistic behaviour by usually...,21276.0,221.0,32.0,biohacking,en-US,2011-06-10,False,What Is Evil? New videos DAILY: https://bigth....,8,True


It is more or less relevant since it contains the vidoes which are contemplation about the longevity and reversing the aging. Bertopic put those vidoes together because they are from the channel called big think, where the scientits come and share their opinions on different kind of discussions. 

In [66]:
#let's check topic # 9
df[df['topic']==9].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
3542,YouTube,jaAY9gKdNKA,https://www.youtube.com/watch?v=jaAY9gKdNKA,L-Theanine + Caffeine Nootropic Stack,To learn more about the information contained ...,ThePromisedWLAN,I don’t take the capsules I always get bulk pr...,37075.0,345.0,47.0,nootropics,unknown,2013-09-24,False,L-Theanine + Caffeine Nootropic Stack To learn...,9,True
5061,YouTube,tvm1ojTWYO0,https://www.youtube.com/watch?v=tvm1ojTWYO0,The Basics of Nootropics,In this video I discuss what defines a nootrop...,Anything Nootropics,,39.0,3.0,0.0,nootropics,en,2015-04-10,False,The Basics of Nootropics In this video I discu...,9,True
6923,YouTube,zc0QLytVNmE,https://www.youtube.com/watch?v=zc0QLytVNmE,Ashwagandha,"In this video, you'll learn the nootropic bene...",NootropicsExpert,Thanks for your videos! Its good to use Ashwag...,109923.0,2815.0,51.0,nootropics,en,2017-04-01,False,"Ashwagandha In this video, you'll learn the no...",9,True
6955,YouTube,J537ll5ckSY,https://www.youtube.com/watch?v=J537ll5ckSY,Coenzyme Q10 (CoQ10),"In this video, you'll learn the nootropic bene...",NootropicsExpert,,84955.0,2211.0,0.0,nootropics,en,2017-04-14,False,"Coenzyme Q10 (CoQ10) In this video, you'll lea...",9,True
6967,YouTube,E3o1DJk-rXY,https://www.youtube.com/watch?v=E3o1DJk-rXY,DHEA,"In this video, you'll learn the nootropic bene...",NootropicsExpert,I am 66 years old and supplemented with 50mg. ...,93125.0,2323.0,128.0,nootropics,en,2017-04-18,False,"DHEA In this video, you'll learn the nootropic...",9,True


Totally relevant, since contains the vidoes abbout the nootropics and their benefits

In [69]:
#let's check topic # 10
df[df['topic']==10].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant
39,YouTube,PL647pg7lW8,https://www.youtube.com/watch?v=PL647pg7lW8,LONGEVITY Welding Forum Contest Winner for Wel...,LONGEVITY Welding offers plasma cutters and TI...,Longevity Welding,,3234.0,3.0,0.0,longevity,en,2010-01-22,False,LONGEVITY Welding Forum Contest Winner for Wel...,10,True
816,YouTube,_p31tr0ZLGA,https://www.youtube.com/watch?v=_p31tr0ZLGA,Longevity TIG WELD 200 Front panel overview,explaining functions longevity-inc.com steveo928.,Stephen Marchio,yes the 5 and 10 sec is your post flow that co...,1641.0,1.0,2.0,longevity,en,2010-11-25,False,Longevity TIG WELD 200 Front panel overview ex...,10,True
910,YouTube,fxGFWIp91e4,https://www.youtube.com/watch?v=fxGFWIp91e4,Longevity TIG WELD 200 With RADNOR electrodes,"nice electrodes, work well with the longevity ...",Stephen Marchio,what happened to Freddy did he pass || Thanks ...,1797.0,6.0,8.0,longevity,en,2010-12-30,False,Longevity TIG WELD 200 With RADNOR electrodes ...,10,True
1054,YouTube,XEgUgByiHEs,https://www.youtube.com/watch?v=XEgUgByiHEs,LONGEVITY StickWeld 140 110/220v Dual Voltage ...,LONGEVITY Welding offers plasma cutters and TI...,Longevity Welding,can some body tell me what is the correct ampe...,6834.0,8.0,10.0,longevity,unknown,2011-02-20,False,LONGEVITY StickWeld 140 110/220v Dual Voltage ...,10,True
1233,YouTube,1khmnYWgqZM,https://www.youtube.com/watch?v=1khmnYWgqZM,Longevity Forcecut 40D plasma cutter 220v test,www.Longevity-inc.com.,Stephen Marchio,You forgot to mention what a pleasure it is wo...,1810.0,3.0,3.0,longevity,unknown,2011-04-18,False,Longevity Forcecut 40D plasma cutter 220v test...,10,True


10 is not relevant at all. It contans the vidoes about the welding tools which belong to (supposedly) to longevity company

In [72]:
i_topics.append(10)

In [76]:
i_topics = list(set(i_topics))

In [78]:
i_topics

[3, 4, 5, 7, 10]

In [88]:
#let's set topics in i_topics as irrelevant in our dataset

tpp.loc[tpp['Topic'].isin(i_topics), 'is_relevant'] = False
df.loc[df['topic'].isin(i_topics), 'is_relevant']=False

In [101]:
#saving the model and topics and probabilites
import numpy as np
import pandas as pd
import pickle


tpp.to_csv('./topics/mpnet_topics.csv', index=False)
topic_model.save("./topics/mpnet_topic_model")
np.save('./topics/topic_probabilities.npy', probs)

preprocessed_data = df['text'].fillna("").tolist()
with open('./topics/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(preprocessed_data, f)

df.to_csv('./topics/final_data.csv', index=False)

#create read.me

2025-08-25 16:52:10,728 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


<h2>Subtopics</h2>

In [97]:
tpp

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio,is_relevant
0,-1,6194,-1_aging_anti_healthy_anti aging,"[aging, anti, healthy, anti aging, healthy agi...",[Dr. Dennis Gross Anti-Aging Scalp Serum John ...,0.706652,True
1,0,5891,0_aging_longevity_rejuvenation_anti,"[aging, longevity, rejuvenation, anti, anti ag...",[McLauren Anti-Aging Sunscreen Spray McLauren ...,0.790528,True
2,1,1987,1_biohacking_biohacker_biohack_dave,"[biohacking, biohacker, biohack, dave, dave as...",[#ShutYourMouth #HostageTape #PatrickMackeown ...,0.770005,True
3,2,1034,2_nootropics_nootropic_cortex_brain,"[nootropics, nootropic, cortex, brain, stack, ...",[Too many Nootropics in one stack for BEGINNER...,0.690522,True
4,3,67,3_klaus_mutant_workout_denmark,"[klaus, mutant, workout, denmark, 2012, super,...",[MUTANT athlete KLAUS RIIS will be at FIBO 201...,0.000000,False
5,4,61,4_dens_lower_song_nootropic,"[dens, lower, song, nootropic, music, record, ...",[Lower Dens - Propagation After just 76 second...,0.377049,False
6,5,52,5_arduino_nootropicdesign_nootropicdesign com_...,"[arduino, nootropicdesign, nootropicdesign com...",[TV Blaster Blast your TV with the Video Exper...,0.038462,False
7,6,37,6_video demonstrating_exercise video_demonstra...,"[video demonstrating, exercise video, demonstr...",[REJUVENATION | TRICEP EXTENSIONS Rejuvenation...,0.675676,True
8,7,35,7_diy_diy bio_bio_pellet,"[diy, diy bio, bio, pellet, water, outside, pr...","[DIY bio pellets Gambitz tank 08June., DIY Bio...",0.085714,False
9,8,34,8_big think_big_https bigth_bigth ink,"[big think, big, https bigth, bigth ink, bigth...",[What is your counsel? New videos DAILY: https...,0.117647,True


Let's run the bertopic analysis on the topics -1, 0,  1, 2. Those are the very large topics and need a further seperation into topics i.e. subtopics

In [110]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

model = SentenceTransformer("paraphrase-mpnet-base-v2") #more nuanced model 

topic_list = [-1, 0, 1, 2]

subtopic_models = {}
subtopic_infos = {}
subtopic_probs = {}
all_subtopics = []
all_indices = []


for topic_id in topic_list: 
    
    mask = df['topic'] == topic_id
    docs_in_topic = df[mask]['text'].fillna("").tolist()
    indices = df[mask].index.tolist()
    
    
    # Custom preprocessing with vectorizer
    vectorizer_model = CountVectorizer(
        stop_words="english",  
        ngram_range=(1, 2),    
        max_df=0.85,           
        min_df=2,              
        max_features=3000  
    )
    
    sub_model = BERTopic(
            embedding_model=model,
            vectorizer_model=vectorizer_model,
            nr_topics="auto",
            top_n_words=10,       
            calculate_probabilities=True,
            min_topic_size=15 
        )
    sub_topics, sub_probs = sub_model.fit_transform(docs_in_topic)
    
    all_subtopics.extend(sub_topics)
    all_indices.extend(indices)

    subtopic_models[topic_id] = sub_model
    subtopic_infos[topic_id] = sub_model.get_topic_info()
    subtopic_probs[topic_id]=sub_probs

In [115]:
df.loc[all_indices, 'subtopic'] = all_subtopics

In [127]:
#let's save the data

import numpy as np

df.to_csv("./topics/subtopics_before/data.csv", index=False)
for i in range(-1, 3):
    subtopic_infos[i].to_csv(f"./topics/subtopics_before/subtopic_{i}.csv", index=False)
    subtopic_models[i].save(f"./topics/subtopics/subtopic_model_{i}")
    np.save(f"./topics/subtopics/topic_probabilities_{i}.npy", subtopic_probs[i])

2025-08-25 18:11:43,817 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2025-08-25 18:11:51,487 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2025-08-25 18:11:55,552 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2025-08-25 18:11:56,254 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the mo

<h3>Manual analysis</h3>

<h4>Analysis of subtopics of topic -1</h4>

In [210]:
import pandas as pd

df = pd.read_csv("./topics/subtopics_before/data.csv", keep_default_na=False)
df['subtopic'] = pd.to_numeric(df['subtopic'], errors='coerce')
df['subtopic'] = df['subtopic'].astype('Int64')
tpp_min_1 = pd.read_csv("./topics/subtopics_before/subtopic_-1.csv")
tpp_min_1

,Topic,Count,Name,Representation,Representative_Docs
0,-1,300,-1_healthy_healthy aging_dr_longevity,"['healthy', 'healthy aging', 'dr', 'longevity'...",['ULTHERAPY | Beverly Hills Rejuvenation Cente...
1,0,5820,0_anti_anti aging_healthy_dr,"['anti', 'anti aging', 'healthy', 'dr', 'rejuv...",['Minimalist 2% Retinoid Anti Aging Night Crea...
2,1,37,1_aging 101_101_georgia_know aging,"['aging 101', '101', 'georgia', 'know aging', ...",['AGING 101: Pt 8 Medications Aging 101 for G...
3,2,19,2_aids_infection_charles_awareness,"['aids', 'infection', 'charles', 'awareness', ...","['Disentangling biological aging, the inflamma..."
4,3,18,3_ministry_longevity_pastor_joseph,"['ministry', 'longevity', 'pastor', 'joseph', ...",['Ron Michalski longevity in ministry Ron is ...


In [212]:
#assigning relevance ratio
def is_video_related(text, keywords):
    text = str(text).lower()
    return any(kw in text for kw in keywords)

def check_relevance_ratio(df, topic_id, subtopic_id, keywords):
    df_topic = df[(df['topic']==topic_id) & (df['subtopic']==subtopic_id)]
    relevant_count = df_topic['text'].apply(lambda x: is_video_related(x, keywords)).sum()
    return relevant_count/len(df_topic)

keywords = ["longevity", "healthy aging", "biohacking", "rejuvenation", "anti-aging", "nootropics", "health"]

In [214]:
tpp_min_1['relevance_ratio'] = tpp_min_1['Topic'].apply(lambda t: check_relevance_ratio(df,-1, t, keywords))

In [216]:
tpp_min_1['is_relevant']=True

In [218]:
tpp_min_1

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio,is_relevant
0,-1,300,-1_healthy_healthy aging_dr_longevity,"['healthy', 'healthy aging', 'dr', 'longevity'...",['ULTHERAPY | Beverly Hills Rejuvenation Cente...,0.676667,True
1,0,5820,0_anti_anti aging_healthy_dr,"['anti', 'anti aging', 'healthy', 'dr', 'rejuv...",['Minimalist 2% Retinoid Anti Aging Night Crea...,0.710309,True
2,1,37,1_aging 101_101_georgia_know aging,"['aging 101', '101', 'georgia', 'know aging', ...",['AGING 101: Pt 8 Medications Aging 101 for G...,0.432432,True
3,2,19,2_aids_infection_charles_awareness,"['aids', 'infection', 'charles', 'awareness', ...","['Disentangling biological aging, the inflamma...",0.368421,True
4,3,18,3_ministry_longevity_pastor_joseph,"['ministry', 'longevity', 'pastor', 'joseph', ...",['Ron Michalski longevity in ministry Ron is ...,0.944444,True


let's check the topics 1, 2, 3

In [220]:
i_subtopics_min=[]

#let's check the topic 1
df[(df['topic']==-1) & (df['subtopic']==1)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
732,YouTube,kDELdRpJ1_U,https://www.youtube.com/watch?v=kDELdRpJ1_U,AGING 101: Pt 14 Home Modifications Housing,Aging 101 for Georgia has been developed and p...,ugagerontology,,932.0,1.0,0.0,well aging,unknown,2010-11-05,False,AGING 101: Pt 14 Home Modifications Housing Ag...,-1,True,1
733,YouTube,mZFWnZ7g86w,https://www.youtube.com/watch?v=mZFWnZ7g86w,AGING 101: Pt 18 Adjusting to Change,Aging 101 for Georgia has been developed and p...,ugagerontology,,4603.0,17.0,0.0,well aging,unknown,2010-11-05,False,AGING 101: Pt 18 Adjusting to Change Aging 101...,-1,True,1
734,YouTube,5LWYHeF7VyI,https://www.youtube.com/watch?v=5LWYHeF7VyI,AGING 101: Pt 16 Caregiving,Aging 101 for Georgia has been developed and p...,ugagerontology,,774.0,0.0,0.0,well aging,unknown,2010-11-05,False,AGING 101: Pt 16 Caregiving Aging 101 for Geor...,-1,True,1
735,YouTube,csQWzuwdC3w,https://www.youtube.com/watch?v=csQWzuwdC3w,AGING 101: Pt 1 Introduction,Aging 101 for Georgia has been developed and p...,ugagerontology,,6790.0,12.0,0.0,healthy aging,unknown,2010-11-05,False,AGING 101: Pt 1 Introduction Aging 101 for Geo...,-1,True,1
736,YouTube,2KvV_ZKBqgI,https://www.youtube.com/watch?v=2KvV_ZKBqgI,AGING 101: Pt 17 Aging Network,Aging 101 for Georgia has been developed and p...,ugagerontology,,491.0,1.0,0.0,well aging,unknown,2010-11-05,False,AGING 101: Pt 17 Aging Network Aging 101 for G...,-1,True,1


this vidoes contains the videos about caregiving the older people. However, our research is not about the caregiving but about reversing the ageing. So this topic is irrelevant

In [222]:
i_subtopics_min.append(1)

#let's check the topic 2
df[(df['topic']==-1) & (df['subtopic']==2)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
543,YouTube,wIhtPcy2bIU,https://www.youtube.com/watch?v=wIhtPcy2bIU,A Conversation on National HIV/AIDS and Aging ...,AIDS.gov spoke with the New York State Departm...,HIV gov,,358.0,0.0,0.0,healthy aging,en,2010-09-01,False,A Conversation on National HIV/AIDS and Aging ...,-1,True,2
1729,YouTube,2YvSyPhDOd4,https://www.youtube.com/watch?v=2YvSyPhDOd4,HIV-Related Stigma and Aging in Sub-Saharan Af...,HIV-related stigma remains the single most imp...,Common Ground Research Networks,,1068.0,1.0,0.0,healthy aging,en,2011-10-31,False,HIV-Related Stigma and Aging in Sub-Saharan Af...,-1,True,2
2121,YouTube,T6ghxQzwopE,https://www.youtube.com/watch?v=T6ghxQzwopE,Aging HIV/AIDS Population Presents Challenges,Being diagnosed with HIV/AIDS used to be a dea...,Voice of America,,545.0,3.0,0.0,healthy aging,unknown,2012-04-06,False,Aging HIV/AIDS Population Presents Challenges ...,-1,True,2
3045,YouTube,5eeEs6R-Nbc,https://www.youtube.com/watch?v=5eeEs6R-Nbc,Charles Emlet - Aging and HIV: Balancing Chall...,This presentation by Dr. Charles Emlet was pre...,TheGilbreaCentre,,133.0,2.0,0.0,well aging,en,2013-03-07,False,Charles Emlet - Aging and HIV: Balancing Chall...,-1,True,2
4244,YouTube,mlq2JWNpdJQ,https://www.youtube.com/watch?v=mlq2JWNpdJQ,HIV and Aging,"Wayne McCormick, MD, MPH - Find this and other...",MWAETC: Project ECHO,,170.0,2.0,0.0,well aging,en,2014-05-29,False,"HIV and Aging Wayne McCormick, MD, MPH - Find ...",-1,True,2


This topic mostly about the HIV in older people and how to prevent it. It can fall under the keyword well aging. So relevent

In [224]:
#let's check the topic 3
df[(df['topic']==-1) & (df['subtopic']==3)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
158,YouTube,8zvz8VQr1G0,https://www.youtube.com/watch?v=8zvz8VQr1G0,Ron Michalski longevity in ministry,Ron is lead pastor at Glad Tidings Church in V...,BC Yukon Pentecostal Assemblies,just joined up myself over at Glad Tidings as ...,1199.0,8.0,2.0,longevity,unknown,2010-03-15,False,Ron Michalski longevity in ministry Ron is le...,-1,True,3
247,YouTube,9NgEGIBdan0,https://www.youtube.com/watch?v=9NgEGIBdan0,Brent Cantelon on longevity (new),Brent Cantelon is the lead pastor at Christian...,BC Yukon Pentecostal Assemblies,,285.0,1.0,0.0,longevity,unknown,2010-04-30,False,Brent Cantelon on longevity (new) Brent Cantel...,-1,True,3
1817,YouTube,givgCXv1uZ8,https://www.youtube.com/watch?v=givgCXv1uZ8,Fred Phelps&#39; Secret of Longevity by Andrew...,Choose Pondering Westboro Baptist Church and F...,Free Press Media Press Inc.,im 12 and wat is this,200.0,1.0,1.0,longevity,en,2011-12-03,False,Fred Phelps&#39; Secret of Longevity by Andrew...,-1,True,3
2977,YouTube,UgDBD3gW41A,https://www.youtube.com/watch?v=UgDBD3gW41A,New Senior Vice President Ken Tankersly on Lon...,,Mason Rutledge,,52.0,0.0,0.0,longevity,en,2013-02-06,False,New Senior Vice President Ken Tankersly on Lon...,-1,True,3
3725,YouTube,SGAVHV_MYoY,https://www.youtube.com/watch?v=SGAVHV_MYoY,"Author Joseph Maroon, M.D. Discusses the Impor...",Learn more about Joseph Maroon at http://autho...,studionowstaging,,2.0,0.0,0.0,longevity,unknown,2013-11-19,False,"Author Joseph Maroon, M.D. Discusses the Impor...",-1,True,3


The topic looks like relevant. So only irrelevant subtopic is 1

In [226]:
i_subtopics_min = list(set(i_subtopics_min))
i_subtopics_min

[1]

In [228]:
tpp_min_1.loc[tpp_min_1['Topic'].isin(i_subtopics_min), 'is_relevant'] = False
df.loc[(df['subtopic'].isin(i_subtopics_min)) & (df['topic']==-1), 'is_relevant']=False

In [230]:
tpp_min_1

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio,is_relevant
0,-1,300,-1_healthy_healthy aging_dr_longevity,"['healthy', 'healthy aging', 'dr', 'longevity'...",['ULTHERAPY | Beverly Hills Rejuvenation Cente...,0.676667,True
1,0,5820,0_anti_anti aging_healthy_dr,"['anti', 'anti aging', 'healthy', 'dr', 'rejuv...",['Minimalist 2% Retinoid Anti Aging Night Crea...,0.710309,True
2,1,37,1_aging 101_101_georgia_know aging,"['aging 101', '101', 'georgia', 'know aging', ...",['AGING 101: Pt 8 Medications Aging 101 for G...,0.432432,False
3,2,19,2_aids_infection_charles_awareness,"['aids', 'infection', 'charles', 'awareness', ...","['Disentangling biological aging, the inflamma...",0.368421,True
4,3,18,3_ministry_longevity_pastor_joseph,"['ministry', 'longevity', 'pastor', 'joseph', ...",['Ron Michalski longevity in ministry Ron is ...,0.944444,True


In [234]:
tpp_min_1.to_csv("./topics/subtopics/subtopic_-1.csv", index=False)

<h4>Analysis of subtopics of topic 0</h4>

In [248]:
tpp_0 = pd.read_csv("./topics/subtopics_before/subtopic_0.csv")

#assing relevance ratio
tpp_0['relevance_ratio'] = tpp_0['Topic'].apply(lambda t: check_relevance_ratio(df,0, t, keywords))

#lets get the topics which has less than the  40 % relevance
tpp_0[tpp_0['relevance_ratio']<=0.40]

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio
16,15,39,15_aging insights_insights_njfa_melissa,"['aging insights', 'insights', 'njfa', 'meliss...",['Aging Insights 7 - Financial Planning In Epi...,0.051282
19,18,34,18_kitv_www kitv_kitv com_kitv aging,"['kitv', 'www kitv', 'kitv com', 'kitv aging',...",['18 10 19 KITV Aging Well Captioned phones ht...,0.117647
20,19,34,19_parents_aging parents_caring_parent,"['parents', 'aging parents', 'caring', 'parent...",['Caring for yourself while caring for aging p...,0.323529
24,23,32,23_shorts_aging shorts_aging society_aging anti,"['shorts', 'aging shorts', 'aging society', 'a...",['Aging Well in an Anti Aging Society Part 3 ...,0.312500
35,34,24,34_pause_pause aging_creative_base,"['pause', 'pause aging', 'creative', 'base', '...",['Base Beauty Creative Agency | Pause Well-Agi...,0.041667
39,38,20,38_lyrics_williams_39 aging_drake,"['lyrics', 'williams', '39 aging', 'drake', 'r...","[""You&#39;re Aging Well Provided to YouTube by...",0.050000


In [254]:
i_subtopics_0=[]

#let's check the topic 15
df[(df['topic']==0) & (df['subtopic']==15)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
1963,YouTube,IadRmENkykw,https://www.youtube.com/watch?v=IadRmENkykw,Aging Insights 1 - Elder Economics and SNAP.,"NJFA's Public Access Cable Show, Aging Insight...",NJ Advocates for Aging Well,,161.0,2.0,0.0,well aging,en,2012-02-03,False,Aging Insights 1 - Elder Economics and SNAP. N...,0,True,15
1972,YouTube,xr5M1rTqaJg,https://www.youtube.com/watch?v=xr5M1rTqaJg,Aging Insights 2 -Creativity Midlife and Beyond,"Episode 2 of Aging Insights, titled Creativity...",NJ Advocates for Aging Well,,104.0,0.0,0.0,well aging,en,2012-02-09,False,Aging Insights 2 -Creativity Midlife and Beyon...,0,True,15
1999,YouTube,rIgMFIOI_nU,https://www.youtube.com/watch?v=rIgMFIOI_nU,Aging Insights 6 - Assitive Devices,"Program Manager, Melissa Chalker is joined by ...",NJ Advocates for Aging Well,,239.0,1.0,0.0,well aging,en,2012-02-24,False,Aging Insights 6 - Assitive Devices Program Ma...,0,True,15
2070,YouTube,qvuhg03-ONw,https://www.youtube.com/watch?v=qvuhg03-ONw,Aging Insights 7 - Financial Planning,"In Episode 7 of Aging Insights, NJFA Executive...",NJ Advocates for Aging Well,,142.0,0.0,0.0,well aging,en,2012-03-22,False,Aging Insights 7 - Financial Planning In Episo...,0,True,15
2140,YouTube,0vnYGc1LP5k,https://www.youtube.com/watch?v=0vnYGc1LP5k,Aging Insights 8 -Your County Office on Aging,"NJFA's Program Manager, Melissa Chalker hosts ...",NJ Advocates for Aging Well,Go Lorraine!!!,305.0,3.0,2.0,well aging,en,2012-04-12,False,Aging Insights 8 -Your County Office on Aging ...,0,True,15


This videos is relevant since contains the videos about aging insights (how to care to be able to live good in older life)

In [261]:

#let's check the topic 15
df[(df['topic']==0) & (df['subtopic']==18)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
7409,YouTube,-tL6IuFBaok,https://www.youtube.com/watch?v=-tL6IuFBaok,Louise Rolland - Aging Well - Medtalks Aug 2017,Louise Rolland from Aging Well Medtalks Aug 20...,Sechelt Hospital Foundation,,52.0,0.0,0.0,well aging,en,2017-09-27,False,Louise Rolland - Aging Well - Medtalks Aug 20...,0,True,18
7413,YouTube,babAjMW77iw,https://www.youtube.com/watch?v=babAjMW77iw,Questions and Answers - Aging Well Medtalks Au...,Questions and Answers from Aging Well Medtalks...,Sechelt Hospital Foundation,Someone ask this woman Torah Kachur/Torah Hunt...,56.0,0.0,1.0,well aging,en,2017-09-27,False,Questions and Answers - Aging Well Medtalks Au...,0,True,18
7426,YouTube,Y9Mzyuii0pU,https://www.youtube.com/watch?v=Y9Mzyuii0pU,Jody Shaw - Aging Well Medtalks Aug 2017,Jody Shaw from Aging Well Medtalks Aug 2017 --...,Sechelt Hospital Foundation,,133.0,2.0,0.0,well aging,en,2017-10-03,False,Jody Shaw - Aging Well Medtalks Aug 2017 Jody ...,0,True,18
7427,YouTube,Lnkdv-iPYZw,https://www.youtube.com/watch?v=Lnkdv-iPYZw,Torah Kachur - Aging Well Medtalks Aug 2017,Torah Kachur from Aging Well Medtalks Aug 2017...,Sechelt Hospital Foundation,Torah Kachur/Hunt claims there is nothing more...,1785.0,21.0,2.0,well aging,en,2017-10-03,False,Torah Kachur - Aging Well Medtalks Aug 2017 To...,0,True,18
8390,YouTube,BitNlIv80O4,https://www.youtube.com/watch?v=BitNlIv80O4,KITV4 Aging Well Ekahi Wellness,"KITV4's Aging Well features 'Ekahi Wellness, a...",Ekahi Wellness,,41.0,1.0,0.0,well aging,en,2018-09-15,False,KITV4 Aging Well Ekahi Wellness KITV4's Aging ...,0,True,18


The vidoes are also relevant

In [269]:
#let's check the topic 38
df[(df['topic']==0) & (df['subtopic']==38)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
219,YouTube,8YjixULJQig,https://www.youtube.com/watch?v=8YjixULJQig,Bad Girlz - Getalittlefizzee,"Badgirlz (Ixindamix & Sim Simmer) - ""It's a fu...",Clatek,23 spi girls good || Parararaaaa🎉🎉🎉 || a good ...,682753.0,7086.0,118.0,nootropics,unknown,2010-04-19,False,Bad Girlz - Getalittlefizzee Badgirlz (Ixindam...,0,True,38
759,YouTube,zQZd-9dwEgQ,https://www.youtube.com/watch?v=zQZd-9dwEgQ,You&#39;re Aging Well,Tupelo Honey.,Pamela Bryan,,390.0,1.0,0.0,well aging,en,2010-11-08,False,You&#39;re Aging Well Tupelo Honey.,0,True,38
1336,YouTube,Ipt9zAuY_yg,https://www.youtube.com/watch?v=Ipt9zAuY_yg,5.27.2011 You&#39;re Aging Well,"Thank you so much, everyone. Huge hugs and big...",Big Is Beauty Project,@giggleclaire &lt;3&lt;#&lt;8 xD || @InnerBeau...,424.0,8.0,7.0,well aging,unknown,2011-05-27,False,5.27.2011 You&#39;re Aging Well Thank you so m...,0,True,38
2058,YouTube,NXtvlg1iw7U,https://www.youtube.com/watch?v=NXtvlg1iw7U,You&#39;re Aging Well,"YOU'RE AGING WELL from ""Written in the Stars: ...",dryadboy,,107.0,0.0,0.0,well aging,unknown,2012-03-17,False,"You&#39;re Aging Well YOU'RE AGING WELL from ""...",0,True,38
4016,YouTube,lPAbE0Ny8yc,https://www.youtube.com/watch?v=lPAbE0Ny8yc,Dar Williams - You&#39;re Aging Well (Bush Hal...,"Dar Williams performing ""You're Aging Well"" at...",apusskidu,A way unrecognized artist. Hear this song on...,4419.0,25.0,7.0,well aging,en,2014-03-06,False,Dar Williams - You&#39;re Aging Well (Bush Hal...,0,True,38


It contains musics so irrelevant. Other subtopics seem like good. So let's only mark as irrelevant the topic 38

In [272]:
i_subtopics_0.append(38)


In [284]:
tpp_0['is_relevant'] = True
tpp_0.loc[tpp_0['Topic'].isin(i_subtopics_0), 'is_relevant'] = False
df.loc[(df['subtopic'].isin(i_subtopics_0)) & (df['topic']==0), 'is_relevant']=False

In [290]:
tpp_0.to_csv("./topics/subtopics/subtopic_0.csv", index=False)

<h4>Analysis of subtopics of topic 1</h4>

In [302]:
tpp_1 = pd.read_csv("./topics/subtopics_before/subtopic_1.csv")

#assing relevance ratio
tpp_1['relevance_ratio'] = tpp_1['Topic'].apply(lambda t: check_relevance_ratio(df,1, t, keywords))

In [304]:
tpp_1

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio
0,-1,1005,-1_health_http_biohack_www,"['health', 'http', 'biohack', 'www', 'amp', 'h...","['Biohack.me and DIYPCR Lepht Anonym, ""Cyberne...",0.751244
1,0,272,0_health_longevity_aging_biohack,"['health', 'longevity', 'aging', 'biohack', 'l...","[""Sleep &amp; Almond Butter - Claudia’s Biohac...",0.926471
2,1,98,1_dave_asprey_dave asprey_bulletproof,"['dave', 'asprey', 'dave asprey', 'bulletproof...",['087: Dave Asprey | Biohacking - is it legit?...,0.826531
3,2,67,2_biohacking biohacking_does_mean_biology,"['biohacking biohacking', 'does', 'mean', 'bio...",['Biohacking Biohacking How to Biohack? What i...,0.970149
4,3,67,3_implant_implants_nfc_microchip,"['implant', 'implants', 'nfc', 'microchip', 'c...",['Magnet Implant Repelling Objects | Jenova Ra...,0.641791
5,4,54,4_biotechnology_biology_synthetic_synthetic bi...,"['biotechnology', 'biology', 'synthetic', 'syn...",['Andrew Hessel: Don’t Fear Synthetic Biology ...,0.240741
6,5,48,5_brain_peptides_biohacking brain_bob,"['brain', 'peptides', 'biohacking brain', 'bob...",['Do Brain Boosters Actually Work? | Paul Fost...,0.750000
7,6,45,6_sleep_braintap_biohacking sleep_better,"['sleep', 'braintap', 'biohacking sleep', 'bet...","['Want to experience less stress,more energy a...",0.733333
8,7,38,7_real_human_trends_effective,"['real', 'human', 'trends', 'effective', 'tren...","[""#1 MISTAKE BIOHACKERS MAKE (And Don’t Even K...",1.000000
9,8,37,8_hacking_bio hacking_bio_body hacking,"['hacking', 'bio hacking', 'bio', 'body hackin...",['Panel - Bio Hacking (on MFSZ2015) In the aft...,0.108108


Let's check topics 4, 7, 9, 12, 13, 14, 15, 16, 17, 18

In [324]:
i_subtopics_1=[]

#let's check the topic 4
df[(df['topic']==1) & (df['subtopic']==4)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
197,YouTube,ehBeMeH69O0,https://www.youtube.com/watch?v=ehBeMeH69O0,TEDxEdmonton - Andrew Hessel - 3/13/10,"""The Rise of Do-It-Yourself Biotechnology"" - G...",TEDx Talks,&quot;We probably need a new type of GMO stand...,2034.0,18.0,4.0,biohacking,unknown,2010-04-05,False,"TEDxEdmonton - Andrew Hessel - 3/13/10 ""The Ri...",1,True,4
202,YouTube,S23owdOuLjc,https://www.youtube.com/watch?v=S23owdOuLjc,Andrew Hessel - The Internet of Living Things,Andrew Hessel reasons that Synthetic Biology w...,momoams,"<a href=""https://www.youtube.com/watch?v=S23ow...",9418.0,103.0,17.0,biohacking,unknown,2010-04-07,False,Andrew Hessel - The Internet of Living Things ...,1,True,4
439,YouTube,u0aVGMSb0tc,https://www.youtube.com/watch?v=u0aVGMSb0tc,"Dr. Nita Farahany - Law, Behavioral Genetics &...","Nita Farahany focuses on the legal, philosophi...",pangea,"Your HEART INTENT is everything, honey. As a m...",6694.0,23.0,5.0,nootropics,unknown,2010-07-17,False,"Dr. Nita Farahany - Law, Behavioral Genetics &...",1,True,4
610,YouTube,cZhHB8cFPo0,https://www.youtube.com/watch?v=cZhHB8cFPo0,Faces of Biotechnology: Jeanne and Aaron,"The Leonardo presents Faces of Biotechnology, ...",The Leonardo,,832.0,2.0,0.0,biohacking,unknown,2010-09-27,False,Faces of Biotechnology: Jeanne and Aaron The L...,1,True,4
626,YouTube,G9T_NKJEdxw,https://www.youtube.com/watch?v=G9T_NKJEdxw,PSCS is bio-curious,Puget Sound Community School teacher Tanya Las...,PSCSvideos,Why does some guy at some foreign talent show ...,431.0,0.0,1.0,biohacking,unknown,2010-10-01,False,PSCS is bio-curious Puget Sound Community Scho...,1,True,4


It seems like relevant

In [327]:
#let's check the topic 7
df[(df['topic']==1) & (df['subtopic']==7)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
6005,YouTube,w3To14yvmr0,https://www.youtube.com/watch?v=w3To14yvmr0,The Upgraded Human | Patric Lanhed | TEDxWHU,The Upgraded Human. What's so fantastic about ...,TEDx Talks,,2471.0,31.0,0.0,biohacking,en,2016-04-04,False,The Upgraded Human | Patric Lanhed | TEDxWHU T...,1,True,7
6069,YouTube,-kw9rcSbrmA,https://www.youtube.com/watch?v=-kw9rcSbrmA,Effective Altruism + Biohacking = Effective Bi...,Effective Altruism + Biohacking = Effective Bi...,Transhuman Videos,,106.0,4.0,0.0,biohacking,en,2016-05-01,False,Effective Altruism + Biohacking = Effective Bi...,1,True,7
6578,YouTube,8Du9jdCzjLw,https://www.youtube.com/watch?v=8Du9jdCzjLw,biohacking V.S biokinesis,PayPal Donate - paypal.me/CodyAustinDalla bioh...,TheBossGOD 1,Can we talk privately || 200 years? That&#39;s...,247.0,6.0,4.0,biohacking,en-US,2016-11-02,False,biohacking V.S biokinesis PayPal Donate - payp...,1,True,7
7200,YouTube,heNPRK_bqXI,https://www.youtube.com/watch?v=heNPRK_bqXI,biohacking biohacking secrets,Visit www.thehabitualman.com and Download your...,"The Man's Guide to Fitness, Money, and Sex.",thanks,408.0,10.0,1.0,biohacking,en,2017-07-12,False,biohacking biohacking secrets Visit www.theh...,1,True,7
7218,YouTube,-fVHvAGMi5Y,https://www.youtube.com/watch?v=-fVHvAGMi5Y,The Dangerous Truth about Biohacking...,gourmetkitchendesign #biohacking #thetruthabou...,The Snack Hacker,,80.0,3.0,0.0,biohacking,en,2017-07-18,False,The Dangerous Truth about Biohacking... gourme...,1,True,7


Relevant as well

In [334]:
#let's check the topic 9
df[(df['topic']==1) & (df['subtopic']==9)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
499,YouTube,dxfDB7iDokQ,https://www.youtube.com/watch?v=dxfDB7iDokQ,Huxley on Huxley - Clip #2,Official Site: http://huxleyonhuxley.com/ Now ...,Cineverse,Does anybody know the name of the song that pl...,1216.0,3.0,1.0,nootropics,unknown,2010-08-11,False,Huxley on Huxley - Clip #2 Official Site: http...,1,True,9
2884,YouTube,s8C3J1QtX6M,https://www.youtube.com/watch?v=s8C3J1QtX6M,Dave at SLF: The Art of Biohacking,In this video I briefly discuss the art and sc...,Bulletproof,this guy works for MONSANTO jajajaj || One of ...,6625.0,83.0,7.0,biohacking,unknown,2012-12-30,False,Dave at SLF: The Art of Biohacking In this vid...,1,True,9
3032,YouTube,KzToC3MwgQA,https://www.youtube.com/watch?v=KzToC3MwgQA,Ophlot &amp; Zx - Biohack,FORTHCOMING YOU SO FAT REC'S Ophlot & Zx-Bioha...,NeurofunkGrid,double drop it with the other stuff? || Over c...,3860.0,86.0,12.0,biohacking,en,2013-03-05,False,Ophlot &amp; Zx - Biohack FORTHCOMING YOU SO F...,1,True,9
4327,YouTube,eh6I7IXiEVM,https://www.youtube.com/watch?v=eh6I7IXiEVM,Real Vegan Cheese - Launch video,The video for the Real Vegan Cheese project's ...,Real Vegan Cheese,It&#39;s a great idea ❤ || OK so this video wa...,49492.0,265.0,63.0,biohacking,en,2014-07-01,False,Real Vegan Cheese - Launch video The video for...,1,True,9
4639,YouTube,_krXs2CsBoU,https://www.youtube.com/watch?v=_krXs2CsBoU,v07 BioHacking Becoming the Best Me I Can Be L...,These are the videos from GrrCON 2014: http://...,Adrian Crenshaw,,379.0,5.0,0.0,biohacking,unknown,2014-10-18,False,v07 BioHacking Becoming the Best Me I Can Be L...,1,True,9


Also relevant. However, the keywords are not

In [337]:
#let's check the topic 14
df[(df['topic']==1) & (df['subtopic']==14)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
538,YouTube,Pdi-7Ox9Fyg,https://www.youtube.com/watch?v=Pdi-7Ox9Fyg,Think Gum at the Pacific Northwest Entrepeneur...,"Matt Davidson, Founder of Think Gum LLC, makes...",thinkgum,,235.0,0.0,0.0,nootropics,unknown,2010-08-30,False,Think Gum at the Pacific Northwest Entrepeneur...,1,True,14
1366,YouTube,3mdX4cxfx0M,https://www.youtube.com/watch?v=3mdX4cxfx0M,Quantified Self Paris: Interview of Emmanuel G...,Emmanuel Gadenne explains the Quantified Self ...,Withings,,281.0,1.0,0.0,biohacking,unknown,2011-06-09,False,Quantified Self Paris: Interview of Emmanuel G...,1,True,14
3531,YouTube,ch2DwaDSP6s,https://www.youtube.com/watch?v=ch2DwaDSP6s,Jari Sarasvuo: Introduction to the ideological...,Introduction to the ideological history of sel...,Quantified Self & Biohacking Finland,,10570.0,30.0,0.0,biohacking,unknown,2013-09-20,False,Jari Sarasvuo: Introduction to the ideological...,1,True,14
3619,YouTube,38xArckxojQ,https://www.youtube.com/watch?v=38xArckxojQ,Teemu Arina: Quantified Self and Biohacking - ...,Introduction to quantified self and biohacking...,Quantified Self & Biohacking Finland,"It&#39;s in Finnish, didn&#39;t know until I s...",1260.0,6.0,1.0,biohacking,unknown,2013-10-17,False,Teemu Arina: Quantified Self and Biohacking - ...,1,True,14
3831,YouTube,JAv7ASyI7kM,https://www.youtube.com/watch?v=JAv7ASyI7kM,Teemu Arina &amp; Lasse Leppäkorpi: Hacking Wh...,Teemu Arina (Biohacker's handbook & Meetin.gs)...,Quantified Self & Biohacking Finland,,1054.0,9.0,0.0,biohacking,unknown,2013-12-17,False,Teemu Arina &amp; Lasse Leppäkorpi: Hacking Wh...,1,True,14


14 is not relevant.

In [339]:
i_subtopics_1.append(14)

In [341]:
#let's check the topic 15
df[(df['topic']==1) & (df['subtopic']==15)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
383,YouTube,hTPVY6ev-pY,https://www.youtube.com/watch?v=hTPVY6ev-pY,CPCO3 - BioHacking (1/2),Esta conferencia trabaja sobre los conceptos y...,Campus Party,,3430.0,15.0,0.0,biohacking,unknown,2010-06-30,False,CPCO3 - BioHacking (1/2) Esta conferencia trab...,1,True,15
2956,YouTube,6cHw0Ur98Ss,https://www.youtube.com/watch?v=6cHw0Ur98Ss,Biotecnología como industria TIC / Biology as ...,"Según Andrew Hessel, fundador de Pink Army Coo...",Fundación Innovación Bankinter,brief yet full of insight.,515.0,7.0,1.0,biohacking,unknown,2013-01-30,False,Biotecnología como industria TIC / Biology as ...,1,True,15
2987,YouTube,OPrnQqlND_o,https://www.youtube.com/watch?v=OPrnQqlND_o,Biohacking: une vie augmentée: Halim Madi at T...,TEDxBordeaux - Terr(hist)oires 2012 Halim est ...,TEDx Talks,Why you didnt do English subtittles? It is eas...,26168.0,442.0,25.0,biohacking,en-US,2013-02-13,False,Biohacking: une vie augmentée: Halim Madi at T...,1,True,15
3688,YouTube,IAFMiOzLugI,https://www.youtube.com/watch?v=IAFMiOzLugI,Hombre biohacker instala sensor en su brazo p...,Hombre biohacker instala sensor en su brazo pa...,EXCELSIOR,,1233.0,2.0,0.0,biohacking,unknown,2013-11-06,False,Hombre biohacker instala sensor en su brazo p...,1,True,15
3700,YouTube,ZBspo40ZizY,https://www.youtube.com/watch?v=ZBspo40ZizY,Un biohacker si impianta un sensore sottopelle...,Per altro ancora visita: http://mcaf.ee/j3xzy.,cinda rella,,19.0,0.0,0.0,biohacking,unknown,2013-11-10,False,Un biohacker si impianta un sensore sottopelle...,1,True,15


It is in different language

In [344]:
i_subtopics_1.append(15)

In [350]:
#let's check the topic 16
df[(df['topic']==1) & (df['subtopic']==16)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
856,YouTube,tNfXOhr0nxI,https://www.youtube.com/watch?v=tNfXOhr0nxI,Airborne Drumz - Biohacker,,KARTOFLATOR,Breakcore\n || JAJAJAJ,2600.0,29.0,2.0,biohacking,en,2010-12-09,False,Airborne Drumz - Biohacker,1,True,16
968,YouTube,sz3PLr-C1T0,https://www.youtube.com/watch?v=sz3PLr-C1T0,BioHacker Symbol Test Animation,"This is a really terrible animation, as you ca...",MrBlueDotStudios,,255.0,0.0,0.0,biohacking,unknown,2011-01-15,False,BioHacker Symbol Test Animation This is a real...,1,True,16
971,YouTube,beiFCJGJNpw,https://www.youtube.com/watch?v=beiFCJGJNpw,BioHacker Proposed Logo,A design I'm working on. Animation is cleaned ...,MrBlueDotStudios,,271.0,3.0,0.0,biohacking,en,2011-01-16,False,BioHacker Proposed Logo A design I'm working o...,1,True,16
1762,YouTube,kmqmpze_-Ok,https://www.youtube.com/watch?v=kmqmpze_-Ok,Beat (BIOHACK),This is my beat I made using soundation.com. I...,Ryan Phillip,,244.0,1.0,0.0,biohacking,unknown,2011-11-11,False,Beat (BIOHACK) This is my beat I made using so...,1,True,16
1766,YouTube,O3Riysz83j0,https://www.youtube.com/watch?v=O3Riysz83j0,Beat Biohack Reverse,This is a beat that I made with soundation.com...,Ryan Phillip,,51.0,2.0,0.0,biohacking,en,2011-11-12,False,Beat Biohack Reverse This is a beat that I mad...,1,True,16


It is not relevant. Contains some musics and not relevant videos

In [353]:
i_subtopics_1.append(16)

In [361]:
#let's check the topic 17
df[(df['topic']==1) & (df['subtopic']==17)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
2614,YouTube,ia-6JmmXX2c,https://www.youtube.com/watch?v=ia-6JmmXX2c,Frequently Asked Questions About Krill IQ,http://krilloil.mercola.com/krill-iq-faq.html?...,Mercola Market,,538.0,7.0,0.0,nootropics,unknown,2012-09-19,False,Frequently Asked Questions About Krill IQ http...,1,True,17
3180,YouTube,hHOsfWX7j_w,https://www.youtube.com/watch?v=hHOsfWX7j_w,WellnessFX Presents: A Fireside Chat with Tim ...,Building the Perfect Human: Tracking Biomarker...,WellnessFX,I love these fireside chats however I wish the...,2241.0,13.0,7.0,biohacking,unknown,2013-04-30,False,WellnessFX Presents: A Fireside Chat with Tim ...,1,True,17
4504,YouTube,fiFoPpRPFNo,https://www.youtube.com/watch?v=fiFoPpRPFNo,Primal Blueprint with Brad Kearns | Primal liv...,Uncensored Interviews with World Leaders in He...,Enterprise Fitness,,174.0,0.0,0.0,longevity,en,2014-09-06,False,Primal Blueprint with Brad Kearns | Primal liv...,1,True,17
5131,YouTube,kuGrAodJ0Jc,https://www.youtube.com/watch?v=kuGrAodJ0Jc,Caleb Jennings: Become the CEO of your own bod...,Visit http://10minutewellnesstips.com/?youtube...,Tips Network,,61.0,2.0,0.0,biohacking,en,2015-05-04,False,Caleb Jennings: Become the CEO of your own bod...,1,True,17
6105,YouTube,djR0t43vu2k,https://www.youtube.com/watch?v=djR0t43vu2k,Andrew Steele on Using Genetic Data for Fitnes...,Show notes: http://biohackersummit.com/2016/05...,HOLOLIFE Summit (Biohacker Summit),"Starts at <a href=""https://www.youtube.com/wat...",644.0,11.0,1.0,biohacking,en,2016-05-18,False,Andrew Steele on Using Genetic Data for Fitnes...,1,True,17


17 is relevant

In [366]:
#let's check the topic 18
df[(df['topic']==1) & (df['subtopic']==18)].head(5)

,source,video_id,video_url,title,description,channel,comments,views,likes,comments_count,keywords,country,date,likes_missing,text,topic,is_relevant,subtopic
4505,YouTube,NEEH-TUcYWc,https://www.youtube.com/watch?v=NEEH-TUcYWc,The Second Annual Bulletproof Biohacker&#39;s ...,http://www.BulletproofConference.com The secon...,Bulletproof,,5222.0,9.0,0.0,biohacking,unknown,2014-09-06,False,The Second Annual Bulletproof Biohacker&#39;s ...,1,True,18
4542,YouTube,_fAtdjwTsOs,https://www.youtube.com/watch?v=_fAtdjwTsOs,Dave Asprey - Biohacking Technologies You&#39;...,The list of speakers for this year's conferenc...,Bulletproof,Look at this guy 8 years later. Should we foll...,31753.0,348.0,19.0,biohacking,unknown,2014-09-15,False,Dave Asprey - Biohacking Technologies You&#39;...,1,True,18
4619,YouTube,IagAAMFSGVo,https://www.youtube.com/watch?v=IagAAMFSGVo,Flow Dojo At The Bulletproof Biohacking Confer...,Here are some scenes from the flow dojo at sec...,Flow Genome Project,How do the guys with the exercise balls get in...,3849.0,34.0,2.0,biohacking,unknown,2014-10-10,False,Flow Dojo At The Bulletproof Biohacking Confer...,1,True,18
5412,YouTube,qSmjeKXvFYY,https://www.youtube.com/watch?v=qSmjeKXvFYY,Wendy Myers - Using Infrared Saunas to Remove ...,Don't miss the biggest biohacking event of the...,Bulletproof,She is a smart woman and she knows stuff. She ...,34138.0,444.0,40.0,biohacking,en,2015-08-18,False,Wendy Myers - Using Infrared Saunas to Remove ...,1,True,18
5426,YouTube,5RsKvUS1R08,https://www.youtube.com/watch?v=5RsKvUS1R08,The 2015 Bulletproof Biohacking Conference,Reserve your place at the biohacking event of ...,Bulletproof,do you think you could somehow incorporate a S...,2524.0,26.0,2.0,biohacking,en,2015-08-25,False,The 2015 Bulletproof Biohacking Conference Res...,1,True,18


it is relevant

In [369]:
i_subtopics_1

[14, 15, 16]

In [371]:
tpp_1['is_relevant'] = True
tpp_1.loc[tpp_1['Topic'].isin(i_subtopics_1), 'is_relevant'] = False
df.loc[(df['subtopic'].isin(i_subtopics_1)) & (df['topic']==1), 'is_relevant']=False

In [377]:
tpp_1.to_csv("./topics/subtopics/subtopic_1.csv", index=False)

<h4>Analysis of subtopics of topic 2</h4>

In [308]:
tpp_2 = pd.read_csv("./topics/subtopics_before/subtopic_2.csv")

#assing relevance ratio
tpp_2['relevance_ratio'] = tpp_2['Topic'].apply(lambda t: check_relevance_ratio(df,2, t, keywords))

In [310]:
tpp_2

,Topic,Count,Name,Representation,Representative_Docs,relevance_ratio
0,-1,17,-1_brainpill_generation_creators_modern,"['brainpill', 'generation', 'creators', 'moder...",['Mindhoney Nootropics: Product Review Mindhon...,0.823529
1,0,241,0_memory_focus_best_natural,"['memory', 'focus', 'best', 'natural', 'boost'...",['Memory Supplements - Quantum Focus™ | Enhanc...,0.717842
2,1,196,1_nootropic stack_stack_livecortex_livecortex com,"['nootropic stack', 'stack', 'livecortex', 'li...",['Ep93: Am I inadequate WITHOUT Nootropics?? O...,0.948980
3,2,153,2_best_video_best nootropics_nootropics video,"['best', 'video', 'best nootropics', 'nootropi...","[""Best Nootropics for 2020 In this video we're...",0.862745
4,3,139,3_fok_coffee_boost_focus,"['fok', 'coffee', 'boost', 'focus', 'energy', ...",['Sweating through workouts? try these #newhis...,0.762590
5,4,115,4_drugs_smart drugs_students_quot,"['drugs', 'smart drugs', 'students', 'quot', '...","['April 1, 2009 - Smart Drugs ', 'Smart Drugs ...",0.052174
6,5,65,5_piracetam_aniracetam_phenylpiracetam_choline,"['piracetam', 'aniracetam', 'phenylpiracetam',...",['1st day on Piracetam I describe my first day...,0.446154
7,6,31,6_adhd_nootropics adhd_add_adderall,"['adhd', 'nootropics adhd', 'add', 'adderall',...",['5 Nootropics Every ADHD Person Should Take I...,0.677419
8,7,30,7_brainhealth nootropics_brainhealth_energy_no...,"['brainhealth nootropics', 'brainhealth', 'ene...","[""Full doses. No Compromises. #fueledbykorvyx ...",0.933333
9,8,28,8_alpha brain_alpha_onnit_rogan,"['alpha brain', 'alpha', 'onnit', 'rogan', 'jo...",['Alpha Brain nootropic Review 2014 Onnit Try ...,0.142857


All the videos seem like relevant. Let's save all of them. 

In [315]:
tpp_2['is_relevant'] = True
tpp_2.to_csv("./topics/subtopics/subtopic_2.csv", index=False)

In [385]:
df[df['is_relevant']==False].shape

(360, 18)

In [395]:
df.shape

(15447, 18)

In [387]:
#let's save the final data with 
df.to_csv("./topics/subtopics/final_data.csv", index=False)

<h1>Summary</h1>

In [407]:
print(f"Relevant topics: {tpp[tpp['is_relevant']==True].shape[0]}")
print(f"Irrelevant topics: {tpp[tpp['is_relevant']==False].shape[0]}")

relevant_sub= tpp_min_1[tpp_min_1['is_relevant']==True].shape[0] + tpp_0[tpp_0['is_relevant']==True].shape[0] + tpp_1[tpp_1['is_relevant']==True].shape[0] +tpp_2[tpp_2['is_relevant']==True].shape[0]
print(f"Relevant subtopics: {relevant_sub}")

irrelevant_sub= tpp_min_1[tpp_min_1['is_relevant']==False].shape[0] + tpp_0[tpp_0['is_relevant']==False].shape[0] + tpp_1[tpp_1['is_relevant']==False].shape[0] +tpp_2[tpp_2['is_relevant']==False].shape[0]
print(f"Irrelevant subtopics: {irrelevant_sub}")

Relevant topics: 7
Irrelevant topics: 5
Relevant subtopics: 73
Irrelevant subtopics: 5


In summary, we have 12 topics and 78 subtopics. In total we have 80 topics
 - 7 relevant topics
 - 5 irrelevant topics
 - 73 relevant subtopics
 - 5 irrelevant topics


To mare irrelevant topics, we created 'is_relevant' column for both data and topic dataframes: if it is "False" then topic and video is not relevant to our video. The relevance was checked by computing how many percentage of the videos contains the keywords. If the percentage was small, the topic was mannually inspected. If the irelevance was found, topic and videos in the topic were marked as 'false' with the help of the column 'is_relevant'.
Out of 15447 videos, 360 was assumed as irrelevant. 

Subtopics were computed only for the first 4 topics, since other topics apart from them were pretty small and specific.